In [1]:

import random
import math

# Delivery locations (x, y coordinates in km)
locations = {
    "Depot": (0, 0),
    "A": (2, 5),
    "B": (5, 2),
    "C": (8, 6),
    "D": (6, 9),
    "E": (1, 8)
}

cities = list(locations.keys())
n = len(cities)

# Calculate distance between two locations
def distance(city1, city2):
    x1, y1 = locations[city1]
    x2, y2 = locations[city2]

    return math.sqrt((x2-x1)**2 + (y2-y1)**2)

# Calculate total delivery route distance
def fitness(route):
    total = 0

    for i in range(len(route)):
        total += distance(
            route[i],
            route[(i+1) % len(route)]
        )

    return total

# Generate a new route using a
# discrete Levy-flight-inspired movement
def levy_flight(route):
    new_route = route.copy()

    # Heavy-tailed random step size
    step = int(random.paretovariate(1.5))
    step = max(1, min(step, len(route)-1))

    # Change the order of locations
    for _ in range(step):
        i, j = random.sample(range(1, len(route)), 2)
        new_route[i], new_route[j] = (
            new_route[j], new_route[i]
        )

    return new_route

# Parameters
nests_count = 20
Pa = 0.25
MaxT = 100

# Initialize random delivery routes
nests = []

for _ in range(nests_count):
    route = cities[1:].copy()
    random.shuffle(route)
    route.insert(0, "Depot")
    nests.append(route)

# Find initial best route
best_route = min(nests, key=fitness)

# Main Cuckoo Search loop
for t in range(MaxT):

    # Generate new cuckoo solutions
    for i in range(nests_count):

        new_route = levy_flight(nests[i])

        # Select a random host nest
        j = random.randrange(nests_count)

        # Replace if new route is better
        if fitness(new_route) < fitness(nests[j]):
            nests[j] = new_route

    # Find and abandon worst nests
    count = max(1, int(Pa * nests_count))

    nests.sort(key=fitness)

    # Replace worst nests with random routes
    for i in range(count):
        new_route = cities[1:].copy()
        random.shuffle(new_route)
        new_route.insert(0, "Depot")
        nests[-(i+1)] = new_route

    # Update best route
    current_best = min(nests, key=fitness)

    if fitness(current_best) < fitness(best_route):
        best_route = current_best.copy()

# Display final result
print("BEST DELIVERY ROUTE:")

print(" -> ".join(best_route + ["Depot"]))

print("Minimum Distance:",
      round(fitness(best_route), 2), "km")


BEST DELIVERY ROUTE:
Depot -> A -> E -> D -> C -> B -> Depot
Minimum Distance: 27.64 km
